# Hidden State 分析

**目的**: 手法選択の前に、LLMのhidden stateに分離可能な構造があるかを確認する。

**分析項目**:
1. 分布の性質（ガウス？ラプラス？尖度は？）
2. 有効次元数（PCA寄与率）
3. エージェント間の相関構造
4. 正解/不正解によるhidden stateの違い
5. 2/3 depth vs 最終層の比較（可能な場合）

In [ ]:
# === Cell 1: セットアップ ===
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('thoughtcomm'):
        !git clone https://github.com/AUMEZAK/thoughtcomm.git
    %cd thoughtcomm
    !pip install -e . -q
    from google.colab import drive
    drive.mount('/content/drive')

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
import json

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
print('Setup done.')

In [ ]:
# === Cell 2: データ読み込み ===
DRIVE_DIR = '/content/drive/MyDrive/thoughtcomm_checkpoints/'

# v1（最終層）のデータを探す
v1_math_path = os.path.join(DRIVE_DIR, 'Qwen_Qwen3-0.6B_math', 'checkpoint.pt')
v1_gsm8k_path = os.path.join(DRIVE_DIR, 'Qwen_Qwen3-0.6B_gsm8k', 'checkpoint.pt')

# v3（2/3 depth）のデータを探す
v3_math_path = os.path.join(DRIVE_DIR, 'Qwen_Qwen3-0.6B_math_hidden_v3.pt')
v3_gsm8k_path = os.path.join(DRIVE_DIR, 'Qwen_Qwen3-0.6B_gsm8k_hidden_v3.pt')

# 利用可能なデータを確認
data = {}
for name, path in [('v1_math', v1_math_path), ('v1_gsm8k', v1_gsm8k_path),
                    ('v3_math', v3_math_path), ('v3_gsm8k', v3_gsm8k_path)]:
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu')
        if 'H' in ckpt:
            data[name] = {'H': ckpt['H'].float(), 'metadata': ckpt.get('metadata', [])}
        elif 'H_list' in ckpt:
            H = torch.stack(ckpt['H_list']).float()
            data[name] = {'H': H, 'metadata': ckpt.get('metadata', [])}
        print(f'{name}: loaded, shape={data[name]["H"].shape}')
    else:
        print(f'{name}: not found at {path}')

# もしパスが違う場合は手動で探す
if not data:
    print('\n=== Searching Drive for hidden state files ===')
    for root, dirs, files in os.walk(DRIVE_DIR):
        for f in files:
            if f.endswith('.pt'):
                print(f'  {os.path.join(root, f)}')

print(f'\nLoaded datasets: {list(data.keys())}')

In [ ]:
# === Cell 3: 基本統計 + 分布分析 ===
# 最初に見つかったデータセットで分析
ds_name = list(data.keys())[0]
H = data[ds_name]['H']
meta = data[ds_name]['metadata']
n_samples, n_h = H.shape
hidden_size = n_h // 3  # 3 agents

print(f'=== 基本統計: {ds_name} ===')
print(f'Shape: {H.shape} ({n_samples} samples, {n_h} dims)')
print(f'  = {n_samples} samples × 3 agents × {hidden_size} hidden_size')
print(f'Mean: {H.mean():.4f}')
print(f'Std:  {H.std():.4f}')
print(f'Min:  {H.min():.4f}')
print(f'Max:  {H.max():.4f}')
print()

# エージェント別統計
for i in range(3):
    Hi = H[:, i*hidden_size:(i+1)*hidden_size]
    print(f'Agent {i}: mean={Hi.mean():.4f}, std={Hi.std():.4f}, '
          f'norm={Hi.norm(dim=1).mean():.2f}')

print()

# 分布の形状分析（全次元のヒストグラム）
H_flat = H.numpy().flatten()
# サンプリング（大きすぎるので100万点まで）
if len(H_flat) > 1000000:
    idx = np.random.choice(len(H_flat), 1000000, replace=False)
    H_sample = H_flat[idx]
else:
    H_sample = H_flat

kurtosis = stats.kurtosis(H_sample)
skewness = stats.skew(H_sample)
print(f'尖度 (kurtosis): {kurtosis:.4f}')
print(f'  ガウス=0, ラプラス=3, >0は裾が重い')
print(f'歪度 (skewness): {skewness:.4f}')

# ガウスとラプラスのフィット比較
_, p_normal = stats.normaltest(H_sample[:5000])  # 5000サンプルで検定
print(f'\n正規性検定 p-value: {p_normal:.2e} ({"ガウスでない" if p_normal < 0.05 else "ガウス的"})')

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ヒストグラム
axes[0].hist(H_sample, bins=200, density=True, alpha=0.7, label='data')
x = np.linspace(H_sample.min(), H_sample.max(), 1000)
axes[0].plot(x, stats.norm.pdf(x, H_sample.mean(), H_sample.std()), 'r-', label='Gaussian fit')
axes[0].plot(x, stats.laplace.pdf(x, H_sample.mean(), H_sample.std()/np.sqrt(2)), 'g-', label='Laplace fit')
axes[0].set_title('Hidden State Distribution')
axes[0].legend()
axes[0].set_xlim(-15, 15)

# Q-Q plot (Gaussian)
stats.probplot(H_sample[:5000], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (vs Gaussian)')

# 各次元の標準偏差の分布
dim_stds = H.std(dim=0).numpy()
axes[2].hist(dim_stds, bins=50, alpha=0.7)
axes[2].axvline(dim_stds.mean(), color='r', linestyle='--', label=f'mean={dim_stds.mean():.2f}')
axes[2].set_title('Per-Dimension Std Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('results/hidden_state_distribution.png', dpi=150)
plt.show()

In [ ]:
# === Cell 4: PCA — 有効次元数 ===
H_np = H.numpy()
H_centered = H_np - H_np.mean(axis=0)

# 全体のPCA
pca = PCA(n_components=min(n_samples, n_h))
pca.fit(H_centered)

cumvar = np.cumsum(pca.explained_variance_ratio_)
eff_dim_90 = np.searchsorted(cumvar, 0.90) + 1
eff_dim_95 = np.searchsorted(cumvar, 0.95) + 1
eff_dim_99 = np.searchsorted(cumvar, 0.99) + 1

print(f'=== PCA分析: {ds_name} ===')
print(f'全次元: {n_h}')
print(f'有効次元数 (90% variance): {eff_dim_90}')
print(f'有効次元数 (95% variance): {eff_dim_95}')
print(f'有効次元数 (99% variance): {eff_dim_99}')
print(f'\n第1主成分の寄与率: {pca.explained_variance_ratio_[0]:.4f}')
print(f'上位10成分の累積寄与率: {cumvar[9]:.4f}')
print(f'上位50成分の累積寄与率: {cumvar[49] if len(cumvar) > 49 else cumvar[-1]:.4f}')

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 累積寄与率
axes[0].plot(range(1, len(cumvar)+1), cumvar, 'b-')
axes[0].axhline(y=0.90, color='r', linestyle=':', label='90%')
axes[0].axhline(y=0.95, color='orange', linestyle=':', label='95%')
axes[0].axhline(y=0.99, color='green', linestyle=':', label='99%')
axes[0].axvline(x=eff_dim_90, color='r', linestyle=':', alpha=0.5)
axes[0].axvline(x=eff_dim_95, color='orange', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title(f'PCA Cumulative Variance (eff_dim_95={eff_dim_95})')
axes[0].legend()
axes[0].set_xlim(0, min(200, len(cumvar)))

# 固有値スペクトル（対数スケール）
axes[1].semilogy(range(1, len(pca.explained_variance_)+1), pca.explained_variance_, 'b-')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Eigenvalue (log scale)')
axes[1].set_title('Eigenvalue Spectrum')
axes[1].set_xlim(0, min(200, len(pca.explained_variance_)))

plt.tight_layout()
plt.savefig('results/hidden_state_pca.png', dpi=150)
plt.show()

In [ ]:
# === Cell 5: エージェント間の相関構造 ===
# 3072 = 3 agents × 1024 に分割
agents = [H[:, i*hidden_size:(i+1)*hidden_size].numpy() for i in range(3)]

# サンプルごとのコサイン類似度
from numpy.linalg import norm

def cosine_sim_batch(A, B):
    """各行のコサイン類似度"""
    dot = (A * B).sum(axis=1)
    nA = norm(A, axis=1)
    nB = norm(B, axis=1)
    return dot / (nA * nB + 1e-8)

cos_01 = cosine_sim_batch(agents[0], agents[1])
cos_02 = cosine_sim_batch(agents[0], agents[2])
cos_12 = cosine_sim_batch(agents[1], agents[2])

print('=== エージェント間コサイン類似度 ===')
print(f'Agent 0 vs 1: mean={cos_01.mean():.4f}, std={cos_01.std():.4f}')
print(f'Agent 0 vs 2: mean={cos_02.mean():.4f}, std={cos_02.std():.4f}')
print(f'Agent 1 vs 2: mean={cos_12.mean():.4f}, std={cos_12.std():.4f}')
print()

# 解釈
avg_cos = np.mean([cos_01.mean(), cos_02.mean(), cos_12.mean()])
if avg_cos > 0.9:
    print('→ エージェント間の類似度が非常に高い（>0.9）。')
    print('  同じモデルの3コピーなので、表現がほぼ同じ。')
    print('  分離すべき「差異」が小さい可能性。')
elif avg_cos > 0.7:
    print('→ エージェント間に中程度の差異がある。')
    print('  分離可能な構造が存在する可能性。')
else:
    print('→ エージェント間の差異が大きい。分離に有利。')

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (cos, title) in zip(axes, [(cos_01, 'Agent 0 vs 1'), 
                                     (cos_02, 'Agent 0 vs 2'),
                                     (cos_12, 'Agent 1 vs 2')]):
    ax.hist(cos, bins=50, alpha=0.7)
    ax.axvline(cos.mean(), color='r', linestyle='--', label=f'mean={cos.mean():.3f}')
    ax.set_title(title)
    ax.set_xlabel('Cosine Similarity')
    ax.legend()
    ax.set_xlim(-1, 1)

plt.tight_layout()
plt.savefig('results/hidden_state_agent_similarity.png', dpi=150)
plt.show()

In [ ]:
# === Cell 6: PCA空間でのエージェント分離可視化 ===
# 3072次元 → 2次元に射影して3エージェントの分布を見る

pca_2d = PCA(n_components=2)
H_2d = pca_2d.fit_transform(H_centered)

# 各サンプルを agent 0/1/2 の部分に分解して個別にPCA
# ただし全体のPCA空間で見る
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左: 全サンプルのPCA散布図（roundで色分け）
# metadataからround情報を取得
rounds = []
for m in meta:
    rounds.append(m.get('round', 0))
rounds = np.array(rounds) if rounds else np.zeros(n_samples)

for r in sorted(set(rounds)):
    mask = rounds == r
    axes[0].scatter(H_2d[mask, 0], H_2d[mask, 1], alpha=0.3, s=10, label=f'Round {int(r)}')
axes[0].set_title(f'PCA 2D Projection (colored by round)')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[0].legend()

# 右: エージェント別のPCA
colors = ['tab:blue', 'tab:orange', 'tab:green']
for i in range(3):
    Hi = agents[i] - agents[i].mean(axis=0)
    pca_agent = PCA(n_components=2)
    Hi_2d = pca_agent.fit_transform(Hi)
    axes[1].scatter(Hi_2d[:, 0], Hi_2d[:, 1], alpha=0.3, s=10, 
                    color=colors[i], label=f'Agent {i}')
axes[1].set_title('Agent-wise PCA (separate projections)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/hidden_state_pca_scatter.png', dpi=150)
plt.show()

In [ ]:
# === Cell 7: 正解/不正解によるhidden stateの違い ===
from evaluation.math_eval import extract_boxed_answer, grade_answer

# metadataから正解/不正解を判定
correct_mask = []
for m in meta:
    if 'answer' in m and 'responses' in m:
        # 各エージェントの最終回答を確認
        any_correct = False
        for resp in m.get('responses', [[]])[-1]:  # 最終ラウンド
            pred = extract_boxed_answer(resp) if resp else None
            if pred and grade_answer(pred, m['answer']):
                any_correct = True
                break
        correct_mask.append(any_correct)
    else:
        correct_mask.append(None)

correct_mask = np.array([c for c in correct_mask if c is not None])
H_valid = H[:len(correct_mask)].numpy()

if len(correct_mask) > 0 and correct_mask.sum() > 0 and (~correct_mask).sum() > 0:
    H_correct = H_valid[correct_mask]
    H_incorrect = H_valid[~correct_mask]
    
    print(f'=== 正解/不正解分析 ===')
    print(f'正解: {correct_mask.sum()}, 不正解: {(~correct_mask).sum()}')
    print(f'\n正解サンプルのnorm:   mean={norm(H_correct, axis=1).mean():.2f}')
    print(f'不正解サンプルのnorm: mean={norm(H_incorrect, axis=1).mean():.2f}')
    
    # エージェント間類似度の比較
    for label, Hx in [('正解', H_correct), ('不正解', H_incorrect)]:
        a0 = Hx[:, :hidden_size]
        a1 = Hx[:, hidden_size:2*hidden_size]
        a2 = Hx[:, 2*hidden_size:]
        cos = cosine_sim_batch(a0, a1).mean()
        print(f'  {label}: Agent 0-1 cos={cos:.4f}')
    
    # PCA空間での分離
    pca_2d = PCA(n_components=2)
    H_all_2d = pca_2d.fit_transform(H_valid - H_valid.mean(axis=0))
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.scatter(H_all_2d[correct_mask, 0], H_all_2d[correct_mask, 1], 
              alpha=0.4, s=15, c='green', label=f'Correct ({correct_mask.sum()})')
    ax.scatter(H_all_2d[~correct_mask, 0], H_all_2d[~correct_mask, 1],
              alpha=0.4, s=15, c='red', label=f'Incorrect ({(~correct_mask).sum()})')
    ax.set_title('PCA: Correct vs Incorrect')
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/hidden_state_correct_vs_incorrect.png', dpi=150)
    plt.show()
else:
    print('正解/不正解の判定ができませんでした（metadataに回答情報がない可能性）')
    print(f'metadata keys: {meta[0].keys() if meta else "empty"}')

In [ ]:
# === Cell 8: ICA分離可能性の直接テスト ===
from sklearn.decomposition import FastICA

print('=== ICA分離テスト（実世界hidden state） ===')
print(f'データ: {ds_name}, shape={H.shape}')

H_np = H.numpy()
H_centered = H_np - H_np.mean(axis=0)
H_std = H_np.std(axis=0)
H_std[H_std < 1e-8] = 1
H_normed = H_centered / H_std

# PCAで次元削減してからICA
for n_comp in [16, 32, 64, 128]:
    if n_comp > min(n_samples, n_h):
        continue
    pca_red = PCA(n_components=n_comp)
    H_pca = pca_red.fit_transform(H_normed)
    var_explained = pca_red.explained_variance_ratio_.sum()
    
    try:
        ica = FastICA(n_components=n_comp, max_iter=1000, random_state=42)
        Z_ica = ica.fit_transform(H_pca)
        
        # ICA成分の尖度（非ガウス性の指標）
        kurtoses = [stats.kurtosis(Z_ica[:, i]) for i in range(n_comp)]
        
        # ICA成分をエージェント帰属（各成分がどのエージェントに最も関連するか）
        # mixing_matrixの行のnormでエージェントブロックへの寄与を見る
        A = ica.mixing_  # (n_features_pca, n_components)
        # PCA逆変換でオリジナル空間に戻す
        A_orig = pca_red.components_.T @ A  # (3072, n_comp)
        
        # 各ICA成分のエージェント別寄与
        agent_contrib = np.zeros((3, n_comp))
        for i in range(3):
            agent_contrib[i] = np.abs(A_orig[i*hidden_size:(i+1)*hidden_size]).sum(axis=0)
        agent_contrib /= agent_contrib.sum(axis=0, keepdims=True) + 1e-8
        
        # 各成分の支配的エージェント
        dominant = agent_contrib.argmax(axis=0)
        dom_counts = [np.sum(dominant == i) for i in range(3)]
        
        # 「私的」成分（1エージェントが>60%寄与）vs「共有」成分
        max_contrib = agent_contrib.max(axis=0)
        private = (max_contrib > 0.6).sum()
        shared = n_comp - private
        
        print(f'\nn_components={n_comp} (var_explained={var_explained:.1%})')
        print(f'  ICA kurtosis: mean={np.mean(kurtoses):.2f}, '
              f'max={np.max(kurtoses):.2f}, min={np.min(kurtoses):.2f}')
        print(f'  Agent dominant: A0={dom_counts[0]}, A1={dom_counts[1]}, A2={dom_counts[2]}')
        print(f'  Private (>60% single agent): {private}/{n_comp}')
        print(f'  Shared (no dominant agent):  {shared}/{n_comp}')
        
    except Exception as e:
        print(f'\nn_components={n_comp}: FAILED ({e})')

In [ ]:
# === Cell 9: 全結果のサマリー ===

print('=' * 60)
print('Hidden State Analysis Summary')
print('=' * 60)

print(f'\nDataset: {ds_name}')
print(f'Shape: {n_samples} samples × {n_h} dims ({n_h//3} per agent × 3 agents)')
print(f'\n1. Distribution: kurtosis={kurtosis:.2f} (0=Gaussian, 3=Laplace)')
print(f'2. Effective dim (95%): {eff_dim_95}/{n_h}')
print(f'3. Agent similarity: mean cosine={avg_cos:.4f}')
print(f'4. ICA separability: see Cell 8 results')

# 結論
print(f'\n=== 結論 ===')
if kurtosis > 1:
    print(f'- 分布は裾が重い（kurtosis={kurtosis:.2f}）→ ICA的分離に有利')
else:
    print(f'- 分布はほぼガウス的 → ICA的分離が難しい可能性')

if eff_dim_95 < n_h * 0.1:
    print(f'- 有効次元が低い（{eff_dim_95}/{n_h}）→ PCA次元削減が有効')
else:
    print(f'- 有効次元が高い → 次元削減で情報が失われる')

if avg_cos > 0.9:
    print(f'- エージェント間が非常に類似（cos={avg_cos:.3f}）→ 分離すべき差異が小さい')
elif avg_cos > 0.5:
    print(f'- エージェント間に中程度の差異（cos={avg_cos:.3f}）→ 分離の余地あり')
else:
    print(f'- エージェント間の差異が大きい（cos={avg_cos:.3f}）→ 分離に有利')

# 保存
analysis_results = {
    'dataset': ds_name,
    'shape': list(H.shape),
    'kurtosis': float(kurtosis),
    'skewness': float(skewness),
    'eff_dim_90': int(eff_dim_90),
    'eff_dim_95': int(eff_dim_95),
    'eff_dim_99': int(eff_dim_99),
    'agent_cosine_01': float(cos_01.mean()),
    'agent_cosine_02': float(cos_02.mean()),
    'agent_cosine_12': float(cos_12.mean()),
    'agent_cosine_mean': float(avg_cos),
}
with open('results/06_hidden_state_analysis.json', 'w') as f:
    json.dump(analysis_results, f, indent=2)
print(f'\nSaved: results/06_hidden_state_analysis.json')